### P-median

In this section, we formulate the p-median problem. We define the following additional input parameters:  
$p$, the number of facilities to be located; $d_j$, the demand of customer $j$; and $c_{ij}$, the cost or weight of serving customer $j$ from facility $i$.

Finally, we define the following decision variables:

$$
y_i =
\begin{cases}
1 & \text{if a facility is located at candidate site } i, \\
0 & \text{otherwise,}
\end{cases}
$$

and $x_{ij}$, the fraction of customer $j$'s demand that is supplied from facility $i$.

With this notation, we can formulate the p-median problem as follows:

$$
\begin{aligned}
\text{min} \quad & \sum_{i \in I} \sum_{j \in J} d_j c_{ij} x_{ij} \\
\text{subject to} \quad & \sum_{i \in I} x_{ij} = 1 \quad \forall j \in J \\
& \sum_{i \in I} y_i = p \\
& x_{ij} - y_i \leq 0 \quad \forall i \in I,\; j \in J \\
& y_i \in \{0, 1\} \quad \forall i \in I \\
& x_{ij} \geq 0 \quad \forall i \in I,\; j \in J
\end{aligned}
$$

In [ ]:
%pip install -q amplpy numpy matplotlib pandas networkx folium
from amplpy import AMPL, ampl_notebook
import numpy as np

# HiGHS is the default. Gurobi requires an AMPL-compatible license.
SOLVER = "highs"  # or "gurobi"
LICENSE_UUID = "default"  # Colab Community Edition; use your UUID locally
runtime = ampl_notebook(modules=[SOLVER], license_uuid=LICENSE_UUID)

def new_ampl():
    return AMPL()

def solve_checked(model):
    model.solve(solver=SOLVER)
    if model.solve_result != "solved":
        raise RuntimeError(f"No proven optimal solution: {model.solve_result}. "
                           "Inspect the solver log before extracting values.")

def values(model, name):
    # Numeric dictionaries keep plotting independent of the solver API.
    return model.var[name].get_values().to_dict()




In [ ]:
# Number of candidate facility sites
num_facilities = 5

# Number of customers
num_customers = 5

# Sets
I = range(num_facilities)
J = range(num_customers)

# Symmetric service cost matrix
c = np.array([[0, 3, 3, 6, 3],
              [3, 0, 4, 5, 5],
              [3, 4, 0, 2, 4],
              [6, 5, 2, 0, 5],
              [3, 5, 4, 5, 0]])

# Customer demand
d = np.array([10, 8, 5, 8, 12])

# Number of facilities to locate
p = 2

In [ ]:
m = new_ampl()
m.eval(r"""
set I;
set J;
param c {I,J};
param d {J} >= 0;
var x {I,J} >= 0;
var y {I} binary;
subject to Assignment {j in J}: sum {i in I} x[i,j] = 1;
param p integer >= 1;
minimize Total_Cost: sum {i in I,j in J} d[j]*c[i,j]*x[i,j];
subject to Cardinality: sum {i in I} y[i] = p;
subject to Link {i in I,j in J}: x[i,j] <= y[i];
""")
m.set["I"] = list(I)
m.set["J"] = list(J)
m.param["c"] = {(i,j): float(c[i,j]) for i in I for j in J}
m.param["d"] = {j: float(d[j]) for j in J}
m.param["p"] = p

solve_checked(m)
x = values(m, "x")
y = values(m, "y")


In [ ]:
if m.solve_result == "solved":
    print("Optimal solution")
    print("Total cost:", m.obj["Total_Cost"].value())
    for i in I:
        if y[i] > 0.1:
            print("Open facility:", i)
else:
    print("Solver status:", m.solve_result)


In [ ]:
import matplotlib.pyplot as plt

def solve_and_plot_p_median(num_facilities=5, num_customers=5, p=2, seed=42,
                            coord_range=(0, 100), metric="euclidean"):

    rng = np.random.default_rng(seed)

    I = range(num_facilities)
    J = range(num_customers)

    facilities_xy = rng.uniform(coord_range[0], coord_range[1], size=(num_facilities, 2))
    customers_xy = rng.uniform(coord_range[0], coord_range[1], size=(num_customers, 2))

    if metric.lower() == "euclidean":
        # c[i,j] = ||facility_i - customer_j||_2
        diff = facilities_xy[:, None, :] - customers_xy[None, :, :]
        c = np.sqrt((diff ** 2).sum(axis=2))
    elif metric.lower() == "manhattan":
        diff = np.abs(facilities_xy[:, None, :] - customers_xy[None, :, :])
        c = diff.sum(axis=2)
    else:
        raise ValueError("metric must be 'euclidean' or 'manhattan'")

    m = new_ampl()
    m.eval(r"""
    set I;
    set J;
    param c {I,J};
    param d {J} >= 0;
    var x {I,J} >= 0;
    var y {I} binary;
    subject to Assignment {j in J}: sum {i in I} x[i,j] = 1;
    param p integer >= 1;
    minimize Total_Cost: sum {i in I,j in J} d[j]*c[i,j]*x[i,j];
    subject to Cardinality: sum {i in I} y[i] = p;
    subject to Link {i in I,j in J}: x[i,j] <= y[i];
    """)
    m.set["I"] = list(I)
    m.set["J"] = list(J)
    m.param["c"] = {(i,j): float(c[i,j]) for i in I for j in J}
    m.param["d"] = {j: 1.0 for j in J}
    m.param["p"] = p
    
    solve_checked(m)
    x = values(m, "x")
    y = values(m, "y")

    min_dist = float("inf")
    max_dist = 0.0

    open_facilities = [i for i in I if y[i] > 0.5]
    assignments = {}  # j -> i (assigned facility)
    for j in J:
        # Since this is an LP assignment, take the argmax of x[i,j]
        i_star = max(I, key=lambda i: x[i, j])
        assignments[j] = i_star
        dist = float(c[i_star, j])
        min_dist = min(min_dist, dist)
        max_dist = max(max_dist, dist)

    print(f"Minimum distance traveled by a customer: {min_dist:.4f}")
    print(f"Maximum distance traveled by a customer: {max_dist:.4f}")
    for i in open_facilities:
        print("Open facility at:", i)

    plt.figure(figsize=(9, 7))

    # Customers
    plt.scatter(customers_xy[:, 0], customers_xy[:, 1], marker="o")
    for j in J:
        plt.annotate(f"C{j}", (customers_xy[j, 0], customers_xy[j, 1]), xytext=(5, 5),
                     textcoords="offset points")

    # Closed facilities
    closed_facilities = [i for i in I if i not in open_facilities]
    if closed_facilities:
        plt.scatter(facilities_xy[closed_facilities, 0], facilities_xy[closed_facilities, 1], marker="s")
        for i in closed_facilities:
            plt.annotate(f"F{i}", (facilities_xy[i, 0], facilities_xy[i, 1]), xytext=(5, 5),
                         textcoords="offset points")

    # Open facilities (highlighted)
    if open_facilities:
        plt.scatter(facilities_xy[open_facilities, 0], facilities_xy[open_facilities, 1], marker="s")
        for i in open_facilities:
            plt.annotate(f"F{i}*", (facilities_xy[i, 0], facilities_xy[i, 1]), xytext=(5, 5),
                         textcoords="offset points")

    # Assignment lines
    for j, i in assignments.items():
        x0, y0 = customers_xy[j, 0], customers_xy[j, 1]
        x1, y1 = facilities_xy[i, 0], facilities_xy[i, 1]
        plt.plot([x0, x1], [y0, y1], linewidth=1)

    plt.title(f"p-median solution (p={p}) in 2D - metric: {metric}")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return m

In [ ]:
solve_and_plot_p_median(num_facilities=12, num_customers=30, p = 4 , seed=1234, metric="euclidean")